In [8]:
import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from sklearn.metrics import accuracy_score, classification_report

In [9]:
non_music_path = "dataset2/non_music"
music_path = "dataset2/music"

In [10]:
non_music_files = [os.path.join(non_music_path, f) for f in os.listdir(non_music_path)]
music_files = [os.path.join(music_path, f) for f in os.listdir(music_path)]

In [11]:
def extract_features(file_path):
    audio, sr = librosa.load(file_path)
    # Extract features like MFCCs, chroma features, etc.
    # For simplicity, let's use only MFCCs for this example
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    return mfccs.flatten()

In [12]:
non_music_features = np.array([extract_features(f) for f in non_music_files])

In [13]:

music_features = np.array([extract_features(f) for f in music_files])

In [14]:
test_music_path ="dataset2/test/music"
test_music_files=[os.path.join(test_music_path,f)for f in os.listdir(test_music_path)]
test_non_music_path ="dataset2/test/non_music"
test_non_music_files=[os.path.join(test_non_music_path,f)for f in os.listdir(test_non_music_path)]

In [15]:
validation_music_path= "dataset2/validation/music"
validation_music_files=[os.path.join(validation_music_path, f) for f in os.listdir(validation_music_path)]
validation_non_music_path= "dataset2/validation/non_music"
validation_non_music_files=[os.path.join(validation_non_music_path, f) for f in os.listdir(validation_non_music_path)]

In [16]:
test_music_features= np.array([extract_features(f) for f in test_music_files])
test_non_music_features= np.array([extract_features(f) for f in test_non_music_files])
validation_music_features= np.array([extract_features(f) for f in validation_music_files])
validation_non_music_features= np.array([extract_features(f) for f in validation_non_music_files])

In [17]:
non_music_labels = np.zeros(len(non_music_files))
music_labels = np.ones(len(music_files))
test_non_music_labels = np.zeros(len(test_non_music_files))
test_music_labels = np.ones(len(test_music_files))

In [18]:
validation_non_music_labels=np.zeros(len(validation_non_music_files))
validation_music_labels=np.ones(len(validation_music_files))

In [19]:
X = np.concatenate((non_music_features, music_features))
y = np.concatenate((non_music_labels, music_labels))

In [20]:
X_train=X
X_test= np.concatenate((test_non_music_features, test_music_features))
y_train=y 
y_test =np.concatenate((test_non_music_labels, test_music_labels))
X_val=np.concatenate((validation_non_music_features, validation_music_features))
y_val=np.concatenate((validation_non_music_labels, validation_music_labels))


In [21]:
model = Sequential([
    Flatten(input_shape=(X_train.shape[1],)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

c:\Users\Amine\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [22]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [23]:
model.fit(X_train, y_train, epochs=10, batch_size=1024, validation_data=(X_val, y_val))

Epoch 1/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 85s 777ms/step - accuracy: 0.8055 - loss: 12.1720 - val_accuracy: 0.8623 - val_loss: 1.2214
Epoch 2/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9578 - loss: 0.2857 - val_accuracy: 0.8741 - val_loss: 0.8636
Epoch 3/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - accuracy: 0.9700 - loss: 0.1490 - val_accuracy: 0.8524 - val_loss: 0.9876
Epoch 4/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 4s 50ms/step - accuracy: 0.9766 - loss: 0.0994 - val_accuracy: 0.9035 - val_loss: 0.5710
Epoch 5/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - accuracy: 0.9741 - loss: 0.1084 - val_accuracy: 0.8865 - val_loss: 0.6860
Epoch 6/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.9830 - loss: 0.0687 - val_accuracy: 0.8558 - val_loss: 0.8734
Epoch 7/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.9877 - loss: 0.0464 - val_accuracy: 0.8637 - val_loss: 0.7973
Epoch 8/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.9873 - loss: 0.0457 - val_accuracy: 0.8728 

In [24]:
y_pred = (model.predict(X_test) > 0.9).astype("int32")
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy: 0.9011


In [25]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

         0.0       0.91      0.89      0.90      5000
         1.0       0.90      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [27]:
model.save("CNNmodelForMp3_90% 100k.h5")